Purpose: Create bed files for input to HOMER motif enrichment.<br>
Date initiated: Feb. 9, 2026<br>
Author: Anna Pardo<br>

From HOMER's documentation: <br>
BED files should have at minimum 6 columns (separated by TABs, additional columns will be ignored)

    Column1: chromosome
    Column2: starting position
    Column3: ending position
    Column4: Unique Peak ID
    Column5: not used
    Column6: Strand (+/- or 0/1, where 0="+", 1="-")


In [1]:
import json
import pandas as pd
import numpy as np
import os

In [2]:
# load gene lists for each genotype
genelists = json.load(open("./cisgenes_bygtbias.json"))

In [2]:
# load GTF files (corrected versions)
yagtf = pd.read_csv("/home/leviathan22/Yucca_genomics/Yaloifolia_all.gtf_corrected.gtf",sep="\t",header=None,comment="#")
yagtf.head()

,0,1,2,3,4,5,6,7,8
0,Chr01,phytozomev13,gene,24835,30805,.,-,.,ID=Yucal.01G000100.v2.1;Name=Yucal.01G000100.1...
1,Chr01,phytozomev13,transcript,24835,30805,.,-,.,ID=Yucal.01G000100.1.v2.1;Parent=Yucal.01G0001...
2,Chr01,phytozomev13,exon,24835,25216,.,-,.,ID=exon-1;Parent=Yucal.01G000100.1.v2.1;gene_i...
3,Chr01,phytozomev13,exon,25501,27311,.,-,.,ID=exon-2;Parent=Yucal.01G000100.1.v2.1;gene_i...
4,Chr01,phytozomev13,exon,30147,30602,.,-,.,ID=exon-3;Parent=Yucal.01G000100.1.v2.1;gene_i...


In [3]:
def gtf_to_bed(filepath):
    gtf = pd.read_csv(filepath,sep="\t",header=None,comment="#")
    genes = gtf[gtf[2]=="gene"]
    
    gids = []
    for i in list(genes[8]):
        gids.append(i.strip().split(";")[0].split("=")[1])
    genes[9] = gids
    
    # create columns with start & stop of 1kb promoter
    ## for reverse-stranded genes: promoter will start from the endpoint (column 4) - i.e. will be to the 'right' of the gene
    pstart = []
    pstop = []
    for i in range(len(genes.index)):
        strand = genes.iloc[i,6]
        if strand=="+":
            refpt = genes.iloc[i,3]-1
            pstart.append(refpt-1001)
            pstop.append(refpt-1)
        elif strand=="-":
            refpt = genes.iloc[i,4]+1
            pstart.append(refpt+1)
            pstop.append(refpt+1001)
            
    genes[10] = pstart
    genes[11] = pstop
    
    bed = genes[[0,10,11,9,2,6]]
    return bed

In [4]:
yabed = gtf_to_bed("/home/leviathan22/Yucca_genomics/Yaloifolia_all.gtf_corrected.gtf")

/tmp/ipykernel_1609/2335614318.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[9] = gids
/tmp/ipykernel_1609/2335614318.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[10] = pstart
/tmp/ipykernel_1609/2335614318.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

In [5]:
yabed.head()

,0,10,11,9,2,6
0,Chr01,30807,31807,Yucal.01G000100.v2.1,gene,-
12,Chr01,82673,83673,Yucal.01G000200.v2.1,gene,-
57,Chr01,51110,52110,Yucal.01G000300.v2.1,gene,-
61,Chr01,151105,152105,Yucal.01G000400.v2.1,gene,+
182,Chr01,191562,192562,Yucal.01G000500.v2.1,gene,+


In [6]:
yfbed = gtf_to_bed("/home/leviathan22/Yucca_genomics/YfilamentosavarC3HAP1v3.1.gene.gtf_corrected.gtf")

/tmp/ipykernel_1609/2335614318.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[9] = gids
/tmp/ipykernel_1609/2335614318.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[10] = pstart
/tmp/ipykernel_1609/2335614318.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

In [7]:
yfbed.head()

,0,10,11,9,2,6
0,Chr01,20533,21533,YufilH1000001m.g,gene,-
6,Chr01,27529,28529,YufilH1000003m.g,gene,-
54,Chr01,91665,92665,YufilH1000007m.g,gene,-
78,Chr01,164742,165742,YufilH1000011m.g,gene,+
98,Chr01,218087,219087,YufilH1000013m.g,gene,+


In [8]:
# save yabed & yfbed
yabed.to_csv("/home/leviathan22/yucca-genomics/Ya_1kb_promoters.bed",sep="\t",header=False,index=False)

In [9]:
yfbed.to_csv("/home/leviathan22/yucca-genomics/Yf_1kb_promoters.bed",sep="\t",header=False,index=False)

In [25]:
# make a function to output (and write) a bed file for a given gene list (as referenced in the 'genelists' dict)
def write_gtbias_bed(gt,bias,topdir,bed):
    if not os.path.exists(os.path.join(topdir,gt)):
        os.makedirs(os.path.join(topdir,gt))
        
    glist = genelists[gt][bias]
    
    subbed = bed[bed[9].isin(glist)]
    
    # write subset bed file
    subbed.to_csv(os.path.join(topdir,gt,gt+"_"+bias+"_promoters_forHOMER.bed"),sep="\t",header=False,index=False)

In [29]:
direc = "/home/leviathan22/yucca-genomics/cisgenes_promoters/"
for k,v in genelists.items():
    for i in v.keys():
        if "Ya" in i:
            write_gtbias_bed(k,i,direc,yabed)
        else:
            write_gtbias_bed(k,i,direc,yfbed)